# Bulgarian ASR Benchmark — Big Models (Google Colab)

**Run this on Google Colab with a T4 GPU** (Runtime > Change runtime type > T4 GPU)

Compares **7 models** on Bulgarian election audio (6 min clip):

| # | Model | Params | Published BG WER | Architecture |
|---|---|---|---|---|
| 1 | nvidia/canary-1b-v2 | 978M | **9.25%** | FastConformer + Transformer |
| 2 | nvidia/parakeet-tdt-0.6b-v3 | 600M | **12.64%** | FastConformer-TDT |
| 3 | openai/whisper-large-v3 | 1.5B | ~15% (est.) | Transformer enc-dec |
| 4 | Groq API (whisper-large-v3) | 1.5B | ~15% (est.) | Same model, cloud LPU |
| 5 | facebook/seamless-m4t-v2-large | 2.3B | Unknown | UnitY2 enc-dec |
| 6 | openai/whisper-large-v3-turbo | 809M | Unknown | Distilled Whisper |
| 7 | sam8000/whisper-large-v3-turbo-bg | 809M | **9.97%** | Fine-tuned for BG |

**Scoring:** Bulgarian char ratio, hallucination detection, content density, repetition, timestamp coverage.

## 1. Setup & Install

In [ ]:
# Install all dependencies
!pip install -q nemo_toolkit[asr] faster-whisper groq transformers accelerate torchaudio sentencepiece

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Set Groq API key (free: https://console.groq.com)
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

## 2. Upload Audio

Upload your `filtered_6to12min.wav` and/or `raw_6to12min.wav` from `output/model_comparison/`.

Or mount Google Drive if you stored files there.

In [ ]:
# Option A: Upload from local machine
from google.colab import files
uploaded = files.upload()  # Select your WAV file(s)

AUDIO_FILES = list(uploaded.keys())
print(f"Uploaded: {AUDIO_FILES}")

In [ ]:
# Option B: Mount Google Drive (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO_FILES = ['/content/drive/MyDrive/filtered_6to12min.wav']

# Option C: Use a test file path directly
# AUDIO_FILES = ['filtered_6to12min.wav']

AUDIO_FILE = AUDIO_FILES[0]
print(f"Using: {AUDIO_FILE} ({os.path.getsize(AUDIO_FILE)/1024/1024:.1f} MB)")

## 3. Transcribe with All Models

Each model runs separately and results are stored for comparison.

In [ ]:
all_results = {}  # model_name -> {text, segments, time}

### 3a. NVIDIA Canary-1B-v2 (best published BG WER: 9.25%)

In [ ]:
import nemo.collections.asr as nemo_asr
import gc

print("Loading nvidia/canary-1b-v2...")
canary = nemo_asr.models.EncDecMultiTaskModel.from_pretrained("nvidia/canary-1b-v2")
canary.eval()

t0 = time.time()
output = canary.transcribe(
    [AUDIO_FILE],
    source_lang="bg",
    target_lang="bg",
    batch_size=1,
    timestamps=True,
)
canary_time = time.time() - t0

canary_segs = []
if hasattr(output[0], 'timestamp') and output[0].timestamp:
    for s in output[0].timestamp.get("segment", []):
        canary_segs.append({"start": round(s["start"], 2), "end": round(s["end"], 2), "text": s["segment"].strip()})
if not canary_segs:
    canary_segs = [{"start": 0, "end": 0, "text": output[0].text}]

all_results["canary_1b"] = {
    "name": "NVIDIA Canary-1B-v2",
    "text": output[0].text,
    "segments": canary_segs,
    "time": canary_time,
}
print(f"  {len(canary_segs)} segments in {canary_time:.1f}s")
print(f"  Preview: {output[0].text[:200]}")

del canary; gc.collect(); torch.cuda.empty_cache()

### 3b. NVIDIA Parakeet-TDT-0.6B-v3 (12.64% WER on BG)

In [ ]:
print("Loading nvidia/parakeet-tdt-0.6b-v3...")
parakeet = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained("nvidia/parakeet-tdt-0.6b-v3")
parakeet.eval()

# Use local attention for longer audio
parakeet.change_attention_model("rel_pos_local_attn", att_context_size=[256, 256])

t0 = time.time()
output = parakeet.transcribe(
    [AUDIO_FILE],
    batch_size=1,
    timestamps=True,
)
parakeet_time = time.time() - t0

parakeet_segs = []
if hasattr(output[0], 'timestamp') and output[0].timestamp:
    for s in output[0].timestamp.get("segment", []):
        parakeet_segs.append({"start": round(s["start"], 2), "end": round(s["end"], 2), "text": s["segment"].strip()})
if not parakeet_segs:
    parakeet_segs = [{"start": 0, "end": 0, "text": output[0].text}]

all_results["parakeet_tdt"] = {
    "name": "NVIDIA Parakeet-TDT-0.6B-v3",
    "text": output[0].text,
    "segments": parakeet_segs,
    "time": parakeet_time,
}
print(f"  {len(parakeet_segs)} segments in {parakeet_time:.1f}s")
print(f"  Preview: {output[0].text[:200]}")

del parakeet; gc.collect(); torch.cuda.empty_cache()

### 3c. Whisper large-v3 (local, faster-whisper on GPU)

In [ ]:
from faster_whisper import WhisperModel

print("Loading whisper-large-v3 (faster-whisper, float16 GPU)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
whisper_model = WhisperModel("large-v3", device=device, compute_type=compute)

t0 = time.time()
segments_iter, info = whisper_model.transcribe(
    AUDIO_FILE,
    language="bg",
    beam_size=5,
    vad_filter=True,
    condition_on_previous_text=False,
)
whisper_segs = [{"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()}
                for s in segments_iter]
whisper_time = time.time() - t0

whisper_text = " ".join(s["text"] for s in whisper_segs)

all_results["whisper_large_v3"] = {
    "name": "Whisper large-v3 (local GPU)",
    "text": whisper_text,
    "segments": whisper_segs,
    "time": whisper_time,
}
print(f"  {len(whisper_segs)} segments in {whisper_time:.1f}s")
print(f"  Preview: {whisper_text[:200]}")

del whisper_model; gc.collect(); torch.cuda.empty_cache()

### 3d. Groq API (Whisper large-v3, cloud benchmark)

In [ ]:
from groq import Groq

groq_key = os.environ.get("GROQ_API_KEY")
if groq_key:
    client = Groq(api_key=groq_key)
    t0 = time.time()
    with open(AUDIO_FILE, "rb") as f:
        resp = client.audio.transcriptions.create(
            file=(AUDIO_FILE, f.read()),
            model="whisper-large-v3",
            language="bg",
            response_format="verbose_json",
            temperature=0.0,
        )
    groq_time = time.time() - t0
    groq_data = resp.model_dump()
    groq_segs = [{"start": s["start"], "end": s["end"], "text": s["text"].strip()}
                 for s in groq_data.get("segments", [])]

    all_results["groq_large_v3"] = {
        "name": "Groq API (large-v3)",
        "text": groq_data.get("text", ""),
        "segments": groq_segs,
        "time": groq_time,
    }
    print(f"  {len(groq_segs)} segments in {groq_time:.1f}s")
    print(f"  Preview: {groq_data.get('text', '')[:200]}")
else:
    print("No GROQ_API_KEY set, skipping Groq benchmark.")

### 3e. Meta SeamlessM4T-v2-Large (2.3B, 101 languages)

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
import torchaudio

print("Loading facebook/seamless-m4t-v2-large...")
seamless_proc = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
seamless_model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    "facebook/seamless-m4t-v2-large",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to("cuda" if torch.cuda.is_available() else "cpu")

audio, sr = torchaudio.load(AUDIO_FILE)
if sr != 16000:
    audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16000)

t0 = time.time()
inputs = seamless_proc(audios=audio.squeeze(0).numpy(), sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to(seamless_model.device) for k, v in inputs.items()}
tokens = seamless_model.generate(**inputs, tgt_lang="bul", generate_speech=False)
seamless_text = seamless_proc.decode(tokens[0].tolist(), skip_special_tokens=True)
seamless_time = time.time() - t0

all_results["seamless_m4t"] = {
    "name": "SeamlessM4T-v2-Large (2.3B)",
    "text": seamless_text,
    "segments": [{"start": 0, "end": 0, "text": seamless_text}],  # no timestamps
    "time": seamless_time,
}
print(f"  Time: {seamless_time:.1f}s")
print(f"  Preview: {seamless_text[:200]}")

del seamless_model, seamless_proc; gc.collect(); torch.cuda.empty_cache()

### 3f. Whisper large-v3-turbo (809M, fast distilled variant)

In [ ]:
print("Loading whisper-large-v3-turbo (faster-whisper, float16 GPU)...")
turbo_model = WhisperModel("large-v3-turbo", device=device, compute_type=compute)

t0 = time.time()
segments_iter, info = turbo_model.transcribe(
    AUDIO_FILE,
    language="bg",
    beam_size=5,
    vad_filter=True,
    condition_on_previous_text=False,
)
turbo_segs = [{"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()}
              for s in segments_iter]
turbo_time = time.time() - t0

turbo_text = " ".join(s["text"] for s in turbo_segs)

all_results["whisper_turbo"] = {
    "name": "Whisper large-v3-turbo (809M)",
    "text": turbo_text,
    "segments": turbo_segs,
    "time": turbo_time,
}
print(f"  {len(turbo_segs)} segments in {turbo_time:.1f}s")
print(f"  Preview: {turbo_text[:200]}")

del turbo_model; gc.collect(); torch.cuda.empty_cache()

### 3g. Fine-tuned Bulgarian Whisper (sam8000, 9.97% WER on FLEURS)

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

BG_MODEL_ID = "sam8000/whisper-large-v3-turbo-bulgarian-bulgaria"
print(f"Loading {BG_MODEL_ID}...")

bg_proc = AutoProcessor.from_pretrained(BG_MODEL_ID)
bg_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    BG_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
).to("cuda" if torch.cuda.is_available() else "cpu")

bg_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=bg_model,
    tokenizer=bg_proc.tokenizer,
    feature_extractor=bg_proc.feature_extractor,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

t0 = time.time()
bg_result = bg_pipe(
    AUDIO_FILE,
    generate_kwargs={"language": "bulgarian", "task": "transcribe"},
    return_timestamps=True,
    chunk_length_s=30,
    batch_size=4,
)
bg_time = time.time() - t0

bg_segs = []
if "chunks" in bg_result:
    for ch in bg_result["chunks"]:
        ts = ch.get("timestamp", (0, 0))
        bg_segs.append({
            "start": ts[0] if ts[0] is not None else 0,
            "end": ts[1] if ts[1] is not None else 0,
            "text": ch.get("text", "").strip(),
        })
if not bg_segs:
    bg_segs = [{"start": 0, "end": 0, "text": bg_result.get("text", "")}]

all_results["bg_finetuned"] = {
    "name": "BG Fine-tuned Whisper (sam8000)",
    "text": bg_result.get("text", ""),
    "segments": bg_segs,
    "time": bg_time,
}
print(f"  {len(bg_segs)} segments in {bg_time:.1f}s")
print(f"  Preview: {bg_result.get('text', '')[:200]}")

del bg_model, bg_proc, bg_pipe; gc.collect(); torch.cuda.empty_cache()

## 4. Correctness Scoring

| Metric | Weight | What it measures |
|---|---|---|
| Bulgarian char ratio | 25% | % Cyrillic vs Latin (higher = less hallucination) |
| Clean text score | 25% | Absence of foreign/garbage characters |
| Content density | 15% | Meaningful text vs silence markers |
| Segment consistency | 15% | Reasonable segmentation for audio length |
| Repetition penalty | 10% | Fewer repeated phrases = better |
| Timestamp coverage | 10% | How much audio is covered |

In [ ]:
import subprocess

# Get audio duration
try:
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-show_entries", "format=duration", "-of", "csv=p=0", AUDIO_FILE],
        capture_output=True, text=True
    )
    AUDIO_DURATION = float(probe.stdout.strip())
except:
    AUDIO_DURATION = 360.0  # default 6 min

print(f"Audio duration: {AUDIO_DURATION:.1f}s ({AUDIO_DURATION/60:.1f} min)")


def score_transcript(name, data):
    text = data["text"]
    segments = data["segments"]
    n_segments = len(segments)

    # Build lines for density check
    lines = [f"[{s['start']:.2f}-{s['end']:.2f}] {s['text']}" for s in segments]

    # 1. BG char ratio (25%)
    cyrillic = len(re.findall(r'[\u0400-\u04FF]', text))
    latin = len(re.findall(r'[a-zA-Z]', text))
    total_alpha = cyrillic + latin
    bg_ratio = cyrillic / max(total_alpha, 1)
    bg_score = min(bg_ratio / 0.95, 1.0)

    # 2. Clean text (25%)
    allowed = len(re.findall(r'[\u0400-\u04FF\s\d.,!?\-\x27\x22()\[\]:;]', text))
    clean_ratio = allowed / max(len(text), 1)
    halluc_score = min(clean_ratio / 0.95, 1.0)

    # 3. Content density (15%)
    content_lines = [l for l in lines if not re.match(r'^\[.*\]\s*[\.…\s]*$', l)]
    density = len(content_lines) / max(n_segments, 1)
    density_score = min(density / 0.80, 1.0)

    # 4. Segment consistency (15%)
    expected_min = max(int(AUDIO_DURATION / 15), 10)
    expected_max = int(AUDIO_DURATION / 2)
    if expected_min <= n_segments <= expected_max:
        segment_score = 1.0
    elif n_segments < expected_min:
        segment_score = max(n_segments / expected_min, 0.1)
    else:
        segment_score = max(1.0 - (n_segments - expected_max) / expected_max, 0.1)

    # 5. Repetition (10%)
    words = text.split()
    if len(words) > 10:
        trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        repeated = sum(c - 1 for c in trigram_counts.values() if c > 2)
        repeat_ratio = repeated / max(len(trigrams), 1)
        repetition_score = max(1.0 - repeat_ratio * 5, 0.0)
    else:
        repetition_score = 0.5

    # 6. Coverage (10%)
    max_time = max((s["end"] for s in segments), default=0)
    coverage = min(max_time / AUDIO_DURATION, 1.0)

    total = (bg_score*0.25 + halluc_score*0.25 + density_score*0.15 +
             segment_score*0.15 + repetition_score*0.10 + coverage*0.10)

    return {
        "model": name, "display": data["name"],
        "total": round(total*100, 1),
        "bg%": round(bg_ratio*100, 1),
        "clean%": round(halluc_score*100, 1),
        "dense%": round(density*100, 1),
        "segs": n_segments,
        "seg_sc": round(segment_score*100, 1),
        "rept%": round(repetition_score*100, 1),
        "cover%": round(coverage*100, 1),
        "time": round(data["time"], 1),
        "chars": len(text),
        "latin": latin,
    }

In [ ]:
# Score all models
scores = [score_transcript(k, v) for k, v in all_results.items()]
scores.sort(key=lambda x: x["total"], reverse=True)

print(f"\n{'Rank':<5} {'Model':<32} {'SCORE':>6} {'BG%':>6} {'Clean%':>7} {'Segs':>5} {'Rept%':>6} {'Cover%':>7} {'Time':>7} {'Latin':>6}")
print("-" * 110)
for i, s in enumerate(scores, 1):
    star = " ***" if i == 1 else ""
    print(f"{i:<5} {s['display']:<32} {s['total']:>5.1f}% {s['bg%']:>5.1f}% {s['clean%']:>6.1f}% {s['segs']:>5} {s['rept%']:>5.1f}% {s['cover%']:>6.1f}% {s['time']:>6.1f}s {s['latin']:>5}{star}")

## 5. Visual Comparison

In [ ]:
print("\n=== CORRECTNESS SCORE ===")
print()
for s in scores:
    bar_len = int(s["total"] / 2)
    bar = "\u2588" * bar_len + "\u2591" * (50 - bar_len)
    best = " << BEST" if s == scores[0] else ""
    print(f"  {s['display']:<32} {bar} {s['total']:>5.1f}%{best}")

print()
print("=== SPEED ===")
print()
max_time = max(s["time"] for s in scores)
for s in sorted(scores, key=lambda x: x["time"]):
    bar_len = max(1, int(s["time"] / max_time * 40))
    print(f"  {s['display']:<32} {'\u2588' * bar_len} {s['time']:.1f}s")

## 6. Side-by-Side Text (First 60s)

## 7b. Generate HTML Dashboard (like NB04)

In [ ]:
import base64

# Encode audio for HTML player
with open(AUDIO_FILE, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")
audio_ext = Path(AUDIO_FILE).suffix.lstrip(".")

# Build KPI cards
best = scores[0]
fastest = min(scores, key=lambda x: x["time"])
most_bg = max(scores, key=lambda x: x["bg%"])
most_segs = max(scores, key=lambda x: x["segs"])

# Build model comparison table rows
model_rows = ""
for i, s in enumerate(scores, 1):
    bar_w = int(s["total"] / 100 * 200)
    medal = ["🥇","🥈","🥉"][i-1] if i <= 3 else f"#{i}"
    row_bg = "#f0fdf4" if i == 1 else "#fff"
    model_rows += f"""<tr style="background:{row_bg}">
      <td style="font-size:18px;text-align:center">{medal}</td>
      <td><strong>{s['display']}</strong></td>
      <td><div style="background:linear-gradient(90deg,#3498db {s['total']}%,#ecf0f1 {s['total']}%);
           height:22px;border-radius:4px;line-height:22px;padding:0 8px;color:#fff;font-size:12px;
           font-weight:bold;min-width:40px">{s['total']}%</div></td>
      <td>{s['bg%']}%</td>
      <td>{s['clean%']}%</td>
      <td>{s['segs']}</td>
      <td>{s['rept%']}%</td>
      <td>{s['cover%']}%</td>
      <td>{s['time']}s</td>
      <td>{s['latin']}</td>
    </tr>"""

# Build transcript comparison
transcript_sections = ""
for key, data in all_results.items():
    s = next((x for x in scores if x["model"] == key), None)
    score_val = s["total"] if s else 0
    segs_html = ""
    for seg in data["segments"][:30]:
        m1, s1 = divmod(seg["start"], 60)
        m2, s2 = divmod(seg["end"], 60)
        segs_html += f'<div class="tseg"><span class="ts">[{int(m1):02d}:{s1:04.1f}-{int(m2):02d}:{s2:04.1f}]</span> {seg["text"]}</div>\n'
    if len(data["segments"]) > 30:
        segs_html += f'<div class="tseg" style="color:#999">... {len(data["segments"])-30} more segments</div>'

    transcript_sections += f"""
    <div class="model-transcript">
      <h3>{data['name']} <span class="badge">{score_val}%</span> <span class="badge-time">{data['time']:.1f}s</span></h3>
      <div class="transcript-box">{segs_html}</div>
    </div>"""

# Build breakdown chart rows
breakdown_html = ""
metrics_list = [
    ("Bulgarian %", "bg%", "#27ae60"),
    ("Clean Text %", "clean%", "#2980b9"),
    ("Density %", "dense%", "#8e44ad"),
    ("Segments OK %", "seg_sc", "#f39c12"),
    ("Low Repetition %", "rept%", "#e74c3c"),
    ("Coverage %", "cover%", "#1abc9c"),
]
for metric_name, key, color in metrics_list:
    bars = ""
    for s in scores:
        w = int(s[key] / 100 * 150)
        bars += f'<div style="display:flex;align-items:center;margin:2px 0"><span style="width:200px;font-size:12px">{s["display"][:25]}</span><div style="background:{color};height:16px;width:{w}px;border-radius:3px;margin-right:6px"></div><span style="font-size:11px">{s[key]}%</span></div>'
    breakdown_html += f'<div class="metric-group"><h4>{metric_name}</h4>{bars}</div>'

# HTML template
html = f"""<!DOCTYPE html>
<html lang="bg">
<head>
<meta charset="UTF-8">
<title>Bulgarian ASR Benchmark Dashboard</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:'Segoe UI',system-ui,sans-serif; background:#f5f6fa; color:#2c3e50; padding:20px; }}
  .header {{ background:linear-gradient(135deg,#2c3e50,#3498db); color:#fff; padding:30px; border-radius:12px; margin-bottom:24px; }}
  .header h1 {{ font-size:28px; margin-bottom:8px; }}
  .header p {{ opacity:0.85; font-size:14px; }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:16px; margin-bottom:24px; }}
  .kpi-card {{ background:#fff; border-radius:10px; padding:20px; text-align:center; box-shadow:0 2px 8px rgba(0,0,0,0.08); }}
  .kpi-label {{ font-size:12px; color:#7f8c8d; text-transform:uppercase; letter-spacing:1px; }}
  .kpi-value {{ font-size:32px; font-weight:700; margin:8px 0; }}
  .kpi-detail {{ font-size:12px; color:#95a5a6; }}
  .section {{ background:#fff; border-radius:10px; padding:24px; margin-bottom:20px; box-shadow:0 2px 8px rgba(0,0,0,0.06); }}
  .section h2 {{ font-size:18px; margin-bottom:16px; color:#2c3e50; border-bottom:2px solid #3498db; padding-bottom:8px; }}
  table {{ width:100%; border-collapse:collapse; font-size:13px; }}
  th {{ background:#f8f9fa; padding:10px 8px; text-align:left; font-weight:600; border-bottom:2px solid #dee2e6; }}
  td {{ padding:8px; border-bottom:1px solid #f0f0f0; }}
  .model-transcript {{ margin-bottom:20px; }}
  .model-transcript h3 {{ font-size:15px; margin-bottom:8px; }}
  .badge {{ background:#27ae60; color:#fff; padding:2px 8px; border-radius:10px; font-size:12px; }}
  .badge-time {{ background:#3498db; color:#fff; padding:2px 8px; border-radius:10px; font-size:12px; }}
  .transcript-box {{ max-height:300px; overflow-y:auto; background:#fafafa; border:1px solid #eee; border-radius:6px; padding:12px; font-size:13px; }}
  .tseg {{ margin:4px 0; line-height:1.5; }}
  .ts {{ color:#7f8c8d; font-size:11px; font-family:monospace; }}
  .metric-group {{ margin-bottom:16px; }}
  .metric-group h4 {{ font-size:13px; color:#555; margin-bottom:4px; }}
  .audio-player {{ margin:16px 0; }}
  .footer {{ text-align:center; color:#95a5a6; font-size:12px; margin-top:30px; padding:20px; }}
</style>
</head>
<body>

<div class="header">
  <h1>Bulgarian ASR Benchmark Dashboard</h1>
  <p>7 models compared on Bulgarian election audio | Generated from Colab GPU benchmark</p>
</div>

<div class="kpi-grid">
  <div class="kpi-card">
    <div class="kpi-label">Best Model</div>
    <div class="kpi-value" style="font-size:20px;color:#27ae60">{best['display']}</div>
    <div class="kpi-detail">Score: {best['total']}%</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Fastest</div>
    <div class="kpi-value" style="font-size:20px;color:#3498db">{fastest['display']}</div>
    <div class="kpi-detail">{fastest['time']}s</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Best BG Accuracy</div>
    <div class="kpi-value" style="font-size:20px;color:#8e44ad">{most_bg['display']}</div>
    <div class="kpi-detail">{most_bg['bg%']}% Cyrillic</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Models Tested</div>
    <div class="kpi-value">{len(scores)}</div>
    <div class="kpi-detail">On {AUDIO_DURATION:.0f}s audio</div>
  </div>
</div>

<div class="section">
  <h2>Audio Player</h2>
  <audio controls style="width:100%">
    <source src="data:audio/{audio_ext};base64,{audio_b64}" type="audio/{audio_ext}">
  </audio>
</div>

<div class="section">
  <h2>Model Ranking</h2>
  <table>
    <tr><th></th><th>Model</th><th>Score</th><th>BG%</th><th>Clean%</th><th>Segs</th><th>Rept%</th><th>Cover%</th><th>Speed</th><th>Latin</th></tr>
    {model_rows}
  </table>
</div>

<div class="section">
  <h2>Score Breakdown</h2>
  {breakdown_html}
</div>

<div class="section">
  <h2>Transcripts (first 30 segments each)</h2>
  {transcript_sections}
</div>

<div class="footer">
  Bulgarian ASR Benchmark | Generated {time.strftime('%Y-%m-%d %H:%M')} | Colab T4 GPU
</div>

</body>
</html>"""

dashboard_path = OUTPUT_DIR / "benchmark_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Dashboard saved: {dashboard_path} ({os.path.getsize(dashboard_path)/1024:.0f} KB)")
print(f"Open: file://{dashboard_path.resolve()}")

# Display inline in Colab
from IPython.display import HTML, display
display(HTML(f'<iframe src="{dashboard_path}" width="100%" height="800px"></iframe>'))

In [ ]:
# Download dashboard + scores
from google.colab import files
files.download(str(OUTPUT_DIR / "benchmark_dashboard.html"))
files.download(str(OUTPUT_DIR / "benchmark_scores.json"))

# Download all TXT transcripts
for f in OUTPUT_DIR.glob("*.txt"):
    files.download(str(f))

## 7. Save Results

In [ ]:
# Save all transcripts
for key, data in all_results.items():
    stem = Path(AUDIO_FILE).stem

    # JSON
    with open(OUTPUT_DIR / f"{stem}_{key}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    # TXT
    with open(OUTPUT_DIR / f"{stem}_{key}.txt", "w", encoding="utf-8") as f:
        for seg in data["segments"]:
            m1, s1 = divmod(seg["start"], 60)
            m2, s2 = divmod(seg["end"], 60)
            f.write(f"[{int(m1):02d}:{s1:05.2f} - {int(m2):02d}:{s2:05.2f}] {seg['text']}\n")

    print(f"  Saved: {stem}_{key}.json/.txt")

# Save scores
with open(OUTPUT_DIR / "benchmark_scores.json", "w", encoding="utf-8") as f:
    json.dump(scores, f, ensure_ascii=False, indent=2)
print(f"\nScores saved to: {OUTPUT_DIR / 'benchmark_scores.json'}")

# List all files
print(f"\nAll output files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name:50s} {f.stat().st_size/1024:.1f} KB")

## 8. Download Results

Download the scores and transcripts back to your local machine.

In [ ]:
# Download scores
from google.colab import files
files.download(str(OUTPUT_DIR / "benchmark_scores.json"))

# Download all TXT transcripts
for f in OUTPUT_DIR.glob("*.txt"):
    files.download(str(f))

## 9. Summary

| # | Model | Params | Published BG WER | Architecture | Runs on 16GB CPU? |
|---|---|---|---|---|---|
| 1 | NVIDIA Canary-1B-v2 | 978M | **9.25%** | FastConformer+Transformer | No (GPU) |
| 2 | BG Fine-tuned (sam8000) | 809M | **9.97%** | Whisper turbo fine-tuned | No (GPU/32GB) |
| 3 | NVIDIA Parakeet-TDT | 600M | **12.64%** | FastConformer-TDT | No (GPU) |
| 4 | Whisper large-v3 | 1.5B | ~15% est. | Transformer enc-dec | Yes (CPU, 7GB) |
| 5 | Groq API (large-v3) | 1.5B | ~15% est. | Same, cloud LPU | Yes (API, free) |
| 6 | SeamlessM4T-v2-Large | 2.3B | Unknown | UnitY2 enc-dec | No (GPU) |
| 7 | Whisper large-v3-turbo | 809M | Unknown | Distilled Whisper | Barely (slow) |

Run this notebook on Colab (free T4 GPU) to benchmark all 7 models on your Bulgarian audio.